# Deep Search RAG System
##Group: Ahmed Haitham, Faisal, Sherif, Yasmine

In [ ]:
!pip install cohere

In [ ]:
import os
import json
import cohere
import requests
from bs4 import BeautifulSoup

# ---------------- Cohere Setup ----------------
os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")  # set this in your environment / .env
co = cohere.Client(os.environ["COHERE_API_KEY"])

# Agent 1  -- Query Rewriter

In [ ]:
# ---------------- Query Expansion ----------------
def expand_query_bilingual(user_query: str):
    """
    Expands a query into:
    - 3 semantic paraphrases in English
    - 3 semantic paraphrases in Arabic
    """
    prompt = f"""
Return only valid JSON. Do not include explanations or text outside JSON.

Task:
1. Paraphrase this query into exactly 3 different natural-language queries in English.
2. Paraphrase this query into exactly 3 different natural-language queries in Arabic.
   Make them semantically equivalent to the English queries, but not literal translations.

User Query: "{user_query}"

Return strictly in this JSON format:
{{
  "queries_en": [
    "english_query1",
    "english_query2",
    "english_query3"
  ],
  "queries_ar": [
    "arabic_query1",
    "arabic_query2",
    "arabic_query3"
  ]
}}
"""
    response = co.chat(model="command-r-plus", message=prompt, temperature=0.8)
    raw_text = response.text.strip()
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        raw_text = raw_text.strip("```json").strip("```")
        try:
            data = json.loads(raw_text)
        except:
            print("⚠️ Could not parse JSON. Raw output:\n", raw_text)
            return None
    return data

# ---------------- Query Cleaning ----------------
def clean_queries_bilingual(queries):
    """
    Removes non-useful/filler words from English or Arabic queries
    while keeping meaning intact.
    """
    queries_str = "\n".join(f"- {q}" for q in queries)
    prompt = f"""
Return only valid JSON. Do not include explanations or text outside JSON.

Task:
1. Take the following list of queries and remove non-useful words such as:
   - filler words ("please", "find", "show", "best way to", etc.)
   - stop words that don't affect product search
   - any country like ("Egypt", "America","UAE", etc.)
2. Keep the query concise and semantically equivalent.

Queries:
{queries_str}

Return in this JSON format:
{{
  "cleaned_queries": [
    "cleaned_query1",
    "cleaned_query2"
  ]
}}
"""
    response = co.chat(model="command-r-plus", message=prompt, temperature=0.5)
    raw_text = response.text.strip()
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        raw_text = raw_text.strip("```json").strip("```")
        data = json.loads(raw_text)
    return data["cleaned_queries"]



# Retrieval Step - Web Scraping

In [ ]:
# ---------------- Amazon Scraper ----------------
def get_product_description_and_category(url, headers=None, timeout=15):
    """Open a product page and extract description + category breadcrumb."""
    headers = headers or {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9"
    }

    resp = requests.get(url, headers=headers, timeout=timeout)
    if resp.status_code != 200:
        return None, None

    soup = BeautifulSoup(resp.text, "html.parser")

    # --- Description ---
    desc = None
    bullets = soup.select("#feature-bullets ul li span")
    if bullets:
        desc = " ".join(b.get_text(strip=True) for b in bullets if b.get_text(strip=True))
    else:
        desc_el = soup.select_one("#productDescription")
        if desc_el:
            desc = desc_el.get_text(strip=True)

    # --- Category (breadcrumb trail) ---
    category = None
    breadcrumb_el = soup.select("#wayfinding-breadcrumbs_feature_div ul li a")
    if breadcrumb_el:
        category = " > ".join(a.get_text(strip=True) for a in breadcrumb_el if a.get_text(strip=True))

    return desc, category

def amazon_search(keyword, k=5, country="com", headers=None, timeout=15):
    base = f"https://www.amazon.{country}/s"
    params = {"k": keyword}
    headers = headers or {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9"
    }
    resp = requests.get(base, params=params, headers=headers, timeout=timeout)
    if resp.status_code != 200:
        raise RuntimeError(f"Bad status {resp.status_code} for URL: {resp.url}")
    text = resp.text.lower()
    if "captcha" in text or "enter the characters" in text or "are you a human" in text:
        raise RuntimeError("Blocked by Amazon (captcha/robot check detected).")

    soup = BeautifulSoup(resp.text, "html.parser")
    items = soup.select("div.s-result-item[data-asin]") or soup.select("div[data-asin]")
    results = []
    for item in items:
        asin = item.get("data-asin", "").strip()
        if not asin:
            continue
        title_el = (
            item.select_one("h2 a span") or
            item.select_one("span.a-size-medium.a-color-base.a-text-normal") or
            item.select_one("a.a-link-normal.a-text-normal span") or
            item.select_one("h2")
        )
        title = title_el.get_text(strip=True) if title_el else None
        link_el = item.select_one("h2 a") or item.select_one("a.a-link-normal.s-no-outline")
        url = None
        if link_el and link_el.get("href"):
            href = link_el["href"]
            url = href if href.startswith("http") else f"https://www.amazon.{country}{href}"
        price_whole = item.select_one("span.a-price-whole")
        price_frac = item.select_one("span.a-price-fraction")
        price = None
        if price_whole:
            price = price_whole.get_text(strip=True)
            if price_frac:
                price = price + "." + price_frac.get_text(strip=True)
        rating_el = item.select_one("span.a-icon-alt")
        rating = rating_el.get_text(strip=True).split(" out of")[0] if rating_el and "out of" in rating_el.get_text() else None
        reviews_el = item.select_one("span.a-size-base.s-underline-text")
        reviews = reviews_el.get_text(strip=True).replace(",", "") if reviews_el else None
        description, category = get_product_description_and_category(url, headers=headers, timeout=timeout) if url else (None, None)


        results.append({
            "keyword": keyword,
            "asin": asin,
            "title": title,
            "description": description,
            "category": category,
            "price": price,
            "rating": rating,
            "reviews": reviews,
            "url": url
            })
        if len(results) >= k:
            break
    return results

def amazon_multi_search(keywords, k=5, country="com", headers=None, timeout=15):
    all_results = []
    for keyword in keywords:
        try:
            results = amazon_search(keyword, k=k, country=country, headers=headers, timeout=timeout)
            all_results.extend(results)
        except Exception as e:
            print(f"Error while searching '{keyword}': {e}")
    return all_results


# Testing Code

In [ ]:
# ---------------- Main Execution ----------------
if __name__ == "__main__":
    user_query = "cheap laptop for school"

    # Step 1: Expand query (Cohere)
    expanded = expand_query_bilingual(user_query)
    if not expanded:
        raise RuntimeError("Failed to expand queries from Cohere.")

    all_queries = expanded["queries_en"] + expanded["queries_ar"]

    # Step 2: Clean queries
    cleaned_queries = clean_queries_bilingual(all_queries)

    print("Cleaned Queries to Search:", cleaned_queries)

    # Step 3: Search Amazon
    results = amazon_multi_search(cleaned_queries, k=3, country="eg")

    print(f"\nCollected {len(results)} products from all queries:")
    for r in results:
        print(f"- {r['title']} | {r['price']} | {r['rating']} stars | {r['url']} | {r['category']}")

#Query & Product-Dictionary Embeddings

In [ ]:
!pip install -q faiss-cpu sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util
import faiss
import numpy as np

# Load multilingual embedding model
model = SentenceTransformer("BAAI/bge-m3")

# Combine product info into one semantic text
def build_product_text(p):
    return f"{p['title']} | {p['description']} | Category: {p['category']}"

product_texts = [build_product_text(p) for p in results]

# 4. Embed products
product_embeddings = model.encode(product_texts, convert_to_numpy=True, normalize_embeddings=True)

# 5. Build FAISS index
d = product_embeddings.shape[1]  # embedding dimensions
index = faiss.IndexFlatIP(d)  # cosine similarity with normalized vectors
index.add(product_embeddings)


# Embed queries
query_embeddings = model.encode(cleaned_queries, convert_to_numpy=True, normalize_embeddings=True)



# Testing Code

In [ ]:
#  Search FAISS for each query
k = 3  # top results per query
for i, q in enumerate(cleaned_queries):
    D, I = index.search(query_embeddings[i:i+1], k)
    print(f"\n🔎 Query: {q}")
    for score, idx in zip(D[0], I[0]):
        print(f"  → {results[idx]['title']}  (score={score:.3f})")


#Agent 2

In [ ]:
# Agent 2: Evaluator

def evaluate_results(products, scores, indices, threshold=0.54):
    """
    Evaluates similarity results.
    Args:
        products: list of product dicts
        scores: similarity scores from FAISS
        indices: indices of products from FAISS
        threshold: minimum score for sufficient relevance
    Returns:
        sufficient_products: list of product dicts above threshold
    """
    sufficient_products = []
    for score, idx in zip(scores, indices):
        if score >= threshold:
            product = products[idx].copy()
            product["score"] = float(score)  # attach score
            sufficient_products.append(product)
    return sufficient_products


# ---- Collect across all queries ----
threshold = 0.54  # adjust depending on precision/recall preference
sufficient_products = []
seen_asins = set()  # to deduplicate

for i, q in enumerate(cleaned_queries):
    D, I = index.search(query_embeddings[i:i+1], k=3)  # FAISS search
    sufficient = evaluate_results(results, D[0], I[0], threshold=threshold)

    for p in sufficient:
        asin = p.get("asin")
        if asin and asin not in seen_asins:   # deduplicate by ASIN
            sufficient_products.append(p)
            seen_asins.add(asin)

# ✅ Final list
if sufficient_products:
    print(f"\n✅ Collected {len(sufficient_products)} unique sufficient products:")
    for p in sufficient_products:
        print(f"  → {p['title']} (ASIN={p['asin']}, score={p['score']:.3f})")
else:
    print("⚠️ No sufficiently relevant results found across all queries.")


# Agent 3

In [ ]:
import cohere
import json
from datetime import datetime

#co = cohere.Client("YOUR_COHERE_API_KEY")

class Agent3Answerer:
    def _init_(self):
        self.history = []  # to keep past Q/A

    def __init__(self):
        # Initialize history so it's always available
        self.history = []

    def format_products(self, products):
        """
        Takes product results and formats them into a readable summary.
        """
        if not products:
            return "⚠️ Sorry, I couldn't find relevant products."

        # Make a structured summary
        summary = "📦 Here are the top products I found:\n\n"
        for i, p in enumerate(products, 1):
            summary += (
                f"{i}. {p.get('title', 'Unknown Title')}\n"
                f"- 💰 Price: {p.get('price', 'N/A')}\n"
                f"- ⭐ Rating: {p.get('rating', 'N/A')} ({p.get('reviews', '0')} reviews)\n"
                f"- 🔗 [View on Amazon]({p.get('url', '#')})\n"
            )
            if p.get("description"):
                summary += f"- 📄 Description: {p['description'][:200]}...\n"
            summary += "\n"
        return summary

    def answer_user(self, user_query, products):
        """
        Uses Cohere to generate a natural language answer,
        while also storing the history.
        """
        formatted = self.format_products(products)

        # Add to history
        self.history.append({
            "timestamp": datetime.now().isoformat(),
            "user_query": user_query,
            "products": products,
            "answer": formatted
        })

        # Use Cohere to polish the final output
        prompt = f"""
You are an AI shopping assistant.
Task:
1. Take the user query and the raw product results.
2. Summarize them into a clear and engaging final answer.
3. Keep it concise, helpful, and structured.

User Query: "{user_query}"

Products Found:
{json.dumps(products, indent=2)}

Base Summary:
{formatted}

Return only the polished final answer in markdown.
"""
        response = co.chat(model="command-r-plus", message=prompt, temperature=0.6)
        return response.text.strip()

    def get_history(self):
        return self.history

# Testing Code

In [ ]:
# ---------------- Example Usage ----------------
if __name__ == "__main__":
    # Mock results (Agent 2’s final filtered output)

    agent3 = Agent3Answerer()
    final_answer = agent3.answer_user(user_query, sufficient_products)

    print("\n--- Final Answer to User ---\n")
    print(final_answer)

    print("\n--- Conversation History ---\n")
    print(agent3.get_history())

# Final Pipeline

In [ ]:
def shopping_pipeline(user_query, max_rounds=3, threshold=0.54, country="eg"):
    """
    Full multi-agent shopping pipeline:
    1. Agent 1 expands/cleans queries (English + Arabic).
    2. Amazon scraper collects products.
    3. Agent 2 evaluates sufficiency using FAISS similarity.
    4. If sufficient, Agent 3 generates polished answer.
       Otherwise, feedback is given and round repeats (up to 3).
    """

    round_count = 0
    agent3 = Agent3Answerer()

    while round_count < max_rounds:
        print(f"\n🔄 Round {round_count+1} for query: {user_query}")

        # ---- Agent 1: Query Expansion ----
        expanded = expand_query_bilingual(user_query)
        if not expanded:
            return "❌ Failed at Agent 1 (query expansion)."

        all_queries = expanded["queries_en"] + expanded["queries_ar"]
        cleaned_queries = clean_queries_bilingual(all_queries)
        print("🧹 Cleaned Queries:", cleaned_queries)

        # ---- Amazon Scraper ----
        results = amazon_multi_search(cleaned_queries, k=3, country=country)
        if not results:
            return "❌ No products retrieved from Amazon."

        # ---- Embed Products ----
        product_texts = [build_product_text(p) for p in results]
        product_embeddings = model.encode(product_texts, convert_to_numpy=True, normalize_embeddings=True)
        d = product_embeddings.shape[1]
        index = faiss.IndexFlatIP(d)
        index.add(product_embeddings)

        # Embed Queries
        query_embeddings = model.encode(cleaned_queries, convert_to_numpy=True, normalize_embeddings=True)

        # ---- Agent 2: Evaluation ----
        sufficient_products = []
        seen_asins = set()

        for i, q in enumerate(cleaned_queries):
            D, I = index.search(query_embeddings[i:i+1], k=3)
            sufficient = evaluate_results(results, D[0], I[0], threshold=threshold)

            for p in sufficient:
                asin = p.get("asin")
                if asin and asin not in seen_asins:
                    sufficient_products.append(p)
                    seen_asins.add(asin)

        if sufficient_products:
            print(f"✅ Found {len(sufficient_products)} sufficient products.")
            # ---- Agent 3: Answer ----
            final_answer = agent3.answer_user(user_query, sufficient_products)
            return final_answer

        else:
            print("⚠️ Agent 2: Insufficient results. Sending feedback to Agent 1...")
            user_query = f"{user_query} (more details: missing attributes, refine search)"
            round_count += 1

    # After max rounds, give up and ask user again
    return "⚠️ Sorry, I couldn’t find enough relevant results. Please try rephrasing your query."


#Testing Code

In [ ]:
shopping_pipeline("cheap laptop for school")

# Gradio

In [ ]:
import gradio as gr

# 🔹 Wrapper for Gradio (adjust variable names if needed)
def shopping_interface(user_query, country, threshold, max_rounds):
    try:
        result = shopping_pipeline(
            user_query=user_query,
            country=country,
            threshold=threshold,
            max_rounds=int(max_rounds),
        )
        return result
    except Exception as e:
        return f"❌ Error: {str(e)}"

# 🔹 Build Gradio UI
def launch_gradio():
    with gr.Blocks(theme="soft") as demo:
        gr.Markdown("## 🛒 Multi-Agent Shopping Assistant (Deep Search RAG)")

        with gr.Row():
            user_query = gr.Textbox(
                label="Enter your shopping query",
                placeholder="e.g. budget-friendly earbuds for travel",
                lines=2
            )

        with gr.Row():
            country = gr.Dropdown(
                choices=["eg", "us", "sa", "ae", "uk"],
                value="eg",
                label="Amazon Country"
            )
            threshold = gr.Slider(
                minimum=0.3,
                maximum=0.9,
                value=0.54,
                step=0.01,
                label="Relevance Threshold"
            )
            max_rounds = gr.Slider(
                minimum=1,
                maximum=5,
                value=3,
                step=1,
                label="Max Rounds"
            )

        output = gr.Textbox(label="Result", lines=10)

        run_btn = gr.Button("🔍 Search")

        run_btn.click(
            fn=shopping_interface,
            inputs=[user_query, country, threshold, max_rounds],
            outputs=output
        )

    demo.launch(share=True)

# 🔹 Run interface
launch_gradio()
